In [222]:
%run ./std_methods.ipynb

In [132]:
import pandas as pd

In [223]:
logger =  initialize_logging()

In [134]:
def generate_date_sequence(start_date,end_date,period):
    
    period = period.lower()
    
    if period == "monthly":
        first_of_starting_month=start_date.strftime('%Y-%m-')+"01"
        date_offset=start_date-datetime.datetime.strptime(first_of_starting_month,'%Y-%m-%d')
        relevant_month_starts =  pd.date_range(start=first_of_starting_month, end=end_date, freq='MS').to_pydatetime().tolist()
        return [ dt + date_offset for dt in relevant_month_starts ]

    if period == "daily":
        return pd.date_range(start=start_date, end=end_date, freq='D').to_pydatetime().tolist()

    if period == "90 days":
        number_of_90_day_periods = math.floor((end_date - start_date).days / 90)
        return [start_date + datetime.timedelta(days=offset_index*90) for offset_index in range(0, number_of_90_day_periods)]

    if period == "biweekly":
        number_of_14_day_periods = math.floor((end_date - start_date).days / 14)
        return [start_date + datetime.timedelta(days=offset_index*14) for offset_index in range(0, number_of_14_day_periods)]

def generate_series_from_item_schedule(budget_data,start_date,num_days):

    end_date = start_date + datetime.timedelta(days=num_days)

    #Monthly
    monthly_items_df=budget_data.loc[budget_data['Cadence'] == "Monthly"]
    monthly_items_list=[]
    for index, row in monthly_items_df.iterrows():
        current_start_date = row["Start Date"]
        current_amount = row["Amount"]
        current_paid_from = row["Paid From"]
        current_priority = row["Priority"]
        current_memo = row["Memo"]

        current_monthly_sequence = generate_date_sequence(current_start_date,end_date,"monthly")

        for dt in current_monthly_sequence:
            monthly_items_list.append([dt,current_amount,current_paid_from,current_priority,current_memo])

    #Daily
    daily_items_df=budget_data.loc[budget_data['Cadence'] == "Daily"]
    daily_items_list=[]
    for index, row in daily_items_df.iterrows():
        current_start_date = row["Start Date"]
        current_amount = row["Amount"]
        current_paid_from = row["Paid From"]
        current_priority = row["Priority"]
        current_memo = row["Memo"]

        current_daily_sequence = generate_date_sequence(current_start_date, end_date, "daily")

        for dt in current_daily_sequence:
            daily_items_list.append([dt,current_amount,current_paid_from,current_priority,current_memo])

    #Bi-weekly
    biweekly_items_df = budget_data.loc[budget_data['Cadence'] == "Bi-weekly"]
    biweekly_items_list = []
    for index, row in biweekly_items_df.iterrows():
        current_start_date = row["Start Date"]
        current_amount = row["Amount"]
        current_paid_from = row["Paid From"]
        current_priority = row["Priority"]
        current_memo = row["Memo"]

        current_biweekly_sequence = generate_date_sequence(current_start_date, end_date, "biweekly")

        for dt in current_biweekly_sequence:
            biweekly_items_list.append([dt, current_amount, current_paid_from, current_priority, current_memo])

    #Once
    once_items_df=budget_data.loc[budget_data['Cadence'] == "Once"]
    once_items_list=[]
    for index, row in once_items_df.iterrows():
        current_start_date = row["Start Date"]
        current_amount = row["Amount"]
        current_paid_from = row["Paid From"]
        current_priority = row["Priority"]
        current_memo = row["Memo"]

        current_once_sequence = [datetime.datetime.strptime(str(current_start_date),'%Y-%m-%d %H:%M:%S')]

        once_items_list.append([current_start_date,current_amount,current_paid_from,current_priority,current_memo])

    #Every 3 months
    every_3_months_items_df=budget_data.loc[budget_data['Cadence'] == "Every 3 months"]
    every_3_months_list=[]
    for index, row in every_3_months_items_df.iterrows():
        current_start_date = row["Start Date"]
        current_amount = row["Amount"]
        current_paid_from = row["Paid From"]
        current_priority = row["Priority"]
        current_memo = row["Memo"]

        current_90_day_sequence = generate_date_sequence(current_start_date, end_date, "90 days")

        for dt in current_90_day_sequence:
            every_3_months_list.append([dt,current_amount,current_paid_from,current_priority,current_memo])

    scheduled_items_df = pd.DataFrame(monthly_items_list + daily_items_list + every_3_months_list + once_items_list + biweekly_items_list)
    scheduled_items_df.columns = ["Date","Amount","Account","Priority","Memo"]
    scheduled_items_df = scheduled_items_df[scheduled_items_df.Date >= start_date]
    return scheduled_items_df.sort_values(by="Date")


def validate_parameter_tables(budget_items_df, 
                              priority_account_rules_df, 
                              priority_item_rules_df, 
                              accounts_df, 
                              milestones_df):
    
    #Since i dont know how to use great ewxpectations all the way, this data type casting will not be reflected in great expectations tests
    budget_items_df['Start Date'] = pd.to_datetime(budget_items_df['Start Date'], format='%Y-%m-%d')
    
    
    #save dfs separately for great_expectations
    budget_items_df.to_csv('/home/hdickie/sandbox/budget_items.csv')
    priority_account_rules_df.to_csv('/home/hdickie/sandbox/priority_account_rules.csv')
    priority_item_rules_df.to_csv('/home/hdickie/sandbox/priority_item_rules.csv')
    accounts_df.to_csv('/home/hdickie/sandbox/accounts.csv')
    milestones_df.to_csv('/home/hdickie/sandbox/milestones.csv')
    
    error_flag = False
    
    ###By sheet:
    
    # Budget Items
    data_source_name = "expense_forecast_budget_items"
    data_source_path = '/home/hdickie/sandbox/budget_items.csv'
    data_asset_name="expense_forecast_budget_items"
    expectation_suite_name="expense_forecast"
    checkpoint_name="validate_parameter_tables"
    context, validator = getValidatorAndUpdatedContext(data_source_name, data_source_path, data_asset_name, expectation_suite_name, checkpoint_name)
    
    #assert table schema
    error_flag = error_flag and not validator.expect_column_to_exist('Start Date').success #these method calls have side effects 
    error_flag = error_flag and not validator.expect_column_to_exist('Priority').success
    error_flag = error_flag and not validator.expect_column_to_exist('Cadence').success
    error_flag = error_flag and not validator.expect_column_to_exist('Amount').success
    error_flag = error_flag and not validator.expect_column_to_exist('Paid From').success
    error_flag = error_flag and not validator.expect_column_to_exist('Paid To').success
    error_flag = error_flag and not validator.expect_column_to_exist('Memo').success
    
    #assert non-nulls
    error_flag = error_flag and not validator.expect_column_values_to_not_be_null('Priority').success
    error_flag = error_flag and not validator.expect_column_values_to_not_be_null('Cadence').success
    error_flag = error_flag and not validator.expect_column_values_to_not_be_null('Amount').success
    
    # assert that all start dates are valid
    error_flag = error_flag and not validator.expect_column_values_to_match_regex('Start Date','[0-9]{4}-(0[1-9]|1[0-2])-(0[1-9]|[12][0-9]|3[01])').success
    
    # assert that all priorities have defined behavior
    # TODO: if new priority levels become athign this will need to be adjusted
    error_flag = error_flag and not validator.expect_column_values_to_be_in_set('Cadence',[1,2,3,4,5,6]).success
    
    # assert that cadence values are in set
    error_flag = error_flag and not validator.expect_column_values_to_be_in_set('Cadence',['once','daily','weekly','monthly','biweekly','yearly','Once','Daily','Weekly','Monthly','Biweekly','Yearly']).success
    
    # assert that amounts are valid floats
    error_flag = error_flag and not validator.expect_column_values_to_be_of_type('Amount','float').success
    
    # assert that paid from/to values are in set
    #TODO: New accounts will need to be added here, possibly dynamically
    error_flag = error_flag and not validator.expect_column_values_to_be_in_set('Paid From',['Checking','Credit 1','Savings']).success
    error_flag = error_flag and not validator.expect_column_values_to_be_in_set('Paid To',['Checking','Credit 1','Savings']).success
    
    #runValidation(context,validator,checkpoint_name,budget_items_df)
    
    # Priority Account Rules
    #assert that all priority indices are integers (this is the source of truth column)
    #assert that account names are in set
    # assert Balance LB/UB are valid floats or +/-Inf
    
    # Priority Item Rules
    #would only need to validate if we were defining new priroity levels which is otuside of scope 
    
    # Accounts
    #assert balances columns are valid floats or +/-Inf
    #assert interest rate is valid float
    #assert interest type is Simple or Compound
    #assert billing_start_dt is valid dt
    
    # Milestones
    #assert account names are in set
    #assert min/max balacne are valid floats or +/-Inf
    
    if error_flag:
        logging.info("Errors were detected. Please check validation docs at file:///home/hdickie/great_expectations/uncommitted/data_docs/local_site/index.html")
    
    return error_flag

def read_budget_item_excel_sheet(excel_path):
    budget_items_df = pd.read_excel(excel_path,sheet_name = 'Budget_Items')
    priority_account_rules_df = pd.read_excel(excel_path,sheet_name = 'Priority_Account_Rules')
    priority_item_rules_df = pd.read_excel(excel_path,sheet_name = 'Priority_Item_Rules')
    accounts_df = pd.read_excel(excel_path,sheet_name = 'Accounts')
    milestones_df = pd.read_excel(excel_path,sheet_name = 'Milestones')
    
    validate_parameter_tables(budget_items_df, priority_account_rules_df, priority_item_rules_df, accounts_df, milestones_df)
    
    return [budget_items_df, priority_account_rules_df, priority_item_rules_df, accounts_df, milestones_df]
    
def write_results(results):
    results.to_csv('/home/hdickie/sandbox/results.csv')
    
def get_initial_balances(accounts_df):
    acct_balances = {}
    for index,row in accounts_df.iterrows():
        acct_balances[row['Account Name']] = row['Current Balance']
    return acct_balances

In [135]:
#execute or defer
def optimize(current_ts_df,scheduled_items_df):
    return current_ts_df

def validate_output(output_df):
    pass


In [244]:
def satisfice(start_date,initial_balances_dict,scheduled_items_df,days_out):
    logger.debug('enter satisfice()')
    logger.debug('start_date:'+str(start_date))
    logger.debug('initial_balances_dict:'+str(initial_balances_dict))
    logger.debug('days_out:'+str(days_out))
    
    #create first row
    initial_row_names = ['Date']
    for key in initial_balances_dict.keys():
        initial_row_names += [key]
    initial_row_names += ['Memo']

    initial_row = [start_date]
    for val in initial_balances_dict.values():
        initial_row += [val]
    initial_row += ['']

    initial_row_df = pd.DataFrame(initial_row).T
    initial_row_df.columns = initial_row_names

    all_days = [start_date + datetime.timedelta(days=d) for d in range(0,days_out)]
    
    for d in all_days:
        if d == start_date:
            previous_rows_df = initial_row_df
            continue

        new_row_df = previous_rows_df.tail(1).copy()
        new_row_df.loc[0,"Date"] = d

        relevant_items_df = scheduled_items_df[ (scheduled_items_df['Date'] == d) & (scheduled_items_df['Priority'] == 1)]
        for index, row in relevant_items_df.iterrows():
            assert sum(new_row_df.columns == row.Account) == 1 #assert there was a matching account
            old_value = new_row_df.loc[:,new_row_df.columns == row.Account]
            new_value = old_value + row.Amount
            new_row_df.loc[:,new_row_df.columns == row.Account] += row.Amount

        previous_rows_df = pd.concat([previous_rows_df,new_row_df])
    satisficed_rows_df = previous_rows_df
    logger.debug('exit satisfice()')
    return satisficed_rows_df


In [245]:
current_ts_df = satisfice(start_date,initial_balances_dict,scheduled_items_df,5)

2022-05-30 16:11:29,596 - Expense_Forecast - DEBUG - enter satisfice()
2022-05-30 16:11:29,599 - Expense_Forecast - DEBUG - start_date:2022-01-01 00:00:00
2022-05-30 16:11:29,600 - Expense_Forecast - DEBUG - initial_balances_dict:{'Checking': 3000.0, 'Savings': 0.0, 'Credit 1': 20000.0, 'Credit 1 Interest': 0.0, 'Loan A': 5000.0, 'Loan A Interest': 0.0, 'Loan B': 4000.0, 'Loan B Interest': 0.0, 'Loan C': 3000.0, 'Loan C Interest': 0.0, 'Loan D': 2000.0, 'Loan D Interest': 0.0, 'Loan E': 1500.0, 'Loan E Interest': 0.0}
2022-05-30 16:11:29,600 - Expense_Forecast - DEBUG - days_out:5
2022-05-30 16:11:29,626 - Expense_Forecast - DEBUG - exit satisfice()


In [250]:
print(scheduled_items_df.to_string())

         Date  Amount   Account  Priority                Memo
0  2022-01-01 -2000.0  Checking         1                Rent
8  2022-01-01   -30.0  Credit 1         1                Food
9  2022-01-02   -30.0  Credit 1         1                Food
10 2022-01-03   -30.0  Credit 1         1                Food
11 2022-01-04   -30.0  Credit 1         1                Food
12 2022-01-05   -30.0  Credit 1         1                Food
13 2022-01-06   -30.0  Credit 1         1                Food
4  2022-01-07   -40.0  Checking         1  Minimum cc payment
14 2022-01-07   -30.0  Credit 1         1                Food
15 2022-01-08   -30.0  Credit 1         1                Food
16 2022-01-09   -30.0  Credit 1         1                Food
17 2022-01-10   -30.0  Credit 1         1                Food
18 2022-01-11   -30.0  Credit 1         1                Food
19 2022-01-12   -30.0  Credit 1         1                Food
20 2022-01-13   -30.0  Credit 1         1                Food
21 2022-

In [248]:
parameter_tables_df = read_budget_item_excel_sheet('/home/hdickie/sandbox/Budget_Items.xlsx')

days_out = 90

start_date = datetime.datetime.strptime('2022-01-01','%Y-%m-%d')
scheduled_items_df = generate_series_from_item_schedule(parameter_tables_df[0],start_date,days_out)

initial_balances_dict = get_initial_balances(parameter_tables_df[3])

current_ts_df = satisfice(start_date,initial_balances_dict,scheduled_items_df,days_out)

output_df = optimize(current_ts_df,scheduled_items_df)

write_results(output_df)

/home/hdickie/.local/lib/python3.10/site-packages/great_expectations/datasource/data_connector/runtime_data_connector.py:133: DeprecationWarning: Specifying batch_identifiers as part of the RuntimeDataConnector config is deprecated as of v0.15.1 and will be removed by v0.18. Please configure batch_identifiers as part of Assets instead.
  warnings.warn(
2022-05-30 16:13:59,876 - Expense_Forecast - DEBUG - enter satisfice()
2022-05-30 16:13:59,877 - Expense_Forecast - DEBUG - start_date:2022-01-01 00:00:00
2022-05-30 16:13:59,877 - Expense_Forecast - DEBUG - initial_balances_dict:{'Checking': 3000.0, 'Savings': 0.0, 'Credit 1': 20000.0, 'Credit 1 Interest': 0.0, 'Loan A': 5000.0, 'Loan A Interest': 0.0, 'Loan B': 4000.0, 'Loan B Interest': 0.0, 'Loan C': 3000.0, 'Loan C Interest': 0.0, 'Loan D': 2000.0, 'Loan D Interest': 0.0, 'Loan E': 1500.0, 'Loan E Interest': 0.0}
2022-05-30 16:13:59,878 - Expense_Forecast - DEBUG - days_out:90


Attempting to instantiate class from config...
	Instantiating as a Datasource, since class_name is Datasource
	Successfully instantiated Datasource


ExecutionEngine class name: PandasExecutionEngine
Data Connectors:
	default_runtime_data_connector_name:RuntimeDataConnector

	Available data_asset_names (0 of 0):
		Note : RuntimeDataConnector will not have data_asset_names until they are passed in through RuntimeBatchRequest

	Unmatched data_references (0 of 0): []

length of batch_request:8
enter get_batch_list_from_batch_request()
length batch_request:8
batch_request:{
  "datasource_name": "expense_forecast_budget_items",
  "data_connector_name": "default_runtime_data_connector_name",
  "data_asset_name": "expense_forecast_budget_items",
  "runtime_parameters": {
    "path": "/home/hdickie/sandbox/budget_items.csv"
  },
  "batch_identifiers": {
    "default_identifier_name": "default_identifier"
  }
}
A: length batch_definition_list1
batch_definition_list[{'datasource_name': 'expense_

2022-05-30 16:14:00,084 - Expense_Forecast - DEBUG - exit satisfice()


In [15]:
ge.get_context().open_data_docs()

/home/hdickie/.local/lib/python3.10/site-packages/great_expectations/datasource/data_connector/runtime_data_connector.py:133: DeprecationWarning: Specifying batch_identifiers as part of the RuntimeDataConnector config is deprecated as of v0.15.1 and will be removed by v0.18. Please configure batch_identifiers as part of Assets instead.
  warnings.warn(
